In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
#import medmnist
#from medmnist import INFO, Evaluator
import tqdm

from torchinfo import summary

import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
from torchvision.datasets import MNIST

import torch.optim as optim
from torcheval.metrics.functional import multiclass_confusion_matrix

import pandas as pd
import numpy as np

from matplotlib import pyplot as plt

In [ ]:
download = True

NUM_EPOCHS = 10
BATCH_SIZE = 16
lr = 0.001



# preprocessing
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5]),
    transforms.Lambda(lambda x: torch.flatten(x))
])

# load the data
dataset = MNIST(root=".", transform=data_transform, download=download)
train_dataset = data.Subset(dataset, torch.arange(40000))
train_dataset_small = data.Subset(dataset, torch.arange(400))
test_dataset = train_dataset = data.Subset(dataset, torch.arange(40000, 60000))


# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE)
train_loader_small = data.DataLoader(dataset=train_dataset_small, batch_size=BATCH_SIZE)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=2*BATCH_SIZE, shuffle=False)

In [ ]:
class DNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.softmax(self.fc3(x), dim=1)
        return x


model = DNN()

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [ ]:
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))

In [ ]:
for epoch in range(100):  # loop over the dataset multiple times

    running_loss = 0.0
    i = 0
    model.train()
    for inputs, labels in tqdm.tqdm(train_loader):
        # get the inputs; data is a list of [inputs, labels]

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        #labels = torch.max(labels, axis=-1).values
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    test(model, "cpu", train_loader)

print('Finished Training')